STEP 1

The Cheat Sheet (Requirement Gathering Principles)

    1) Focus on the problem, not the visuals: Never start a conversation by asking, "What charts do you want to see?". Always ask: "What business problem are you currently facing?".

    2) Define operational expectations: Once the problem is defined, ask what exactly the user expects from the tool and what specific action they will take when they see the data.

    3) Propose an initial solution (Prototyping): Don't build the final report blindly. Sketch an initial layout (even on paper) and make sure it actually solves the business need.

    4) Define the timeframe and frequency: Establish the time context – are decisions made daily, or strategically once a quarter?

    5) Force specific metrics (Prioritization): Limit the number of key indicators to 3-5, instead of agreeing to "show all the data on one screen".

The Interrogation (Stakeholder Questions)
For the Director (Strategic Perspective):

    1. What are the 3 most important strategic goals for the company this year?
    2. By looking at which specific KPIs will you know that we are heading in the right direction and succeeding?
    3. What recent business decision did you have to make blindly because you lacked the right data?
    4. During which meetings will this report be used and presented?

For the Operations Manager (Tactical/Operational Perspective):

    5. What specific problem are you dealing with, and which process generates the most customer complaints?
    6. What exactly do you need to see on your screen every morning to catch a problem before it escalates?
    7. How do you currently track these operations, and what annoys you the most about this process?
    8. When you see a metric deteriorating, what is your first action, and who do you call?

Why is it dangerous to put "all metrics on one screen"?

It is simply unnecessary and causes the data to blend together into visual noise. A dashboard shouldn't just exist for the sake of passively checking numbers. Its primary purpose is to deliver tangible value and drive beneficial business outcomes. Cluttering the screen prevents users from extracting that value.

What is the difference between a Metric and an Actionable Insight?

A metric is just a standalone "number" that we see on a dashboard. An actionable insight, on the other hand, is ready-to-use information deduced from those metrics and KPIs, designed specifically to trigger a further action that drives business development or solves a problem.

STEP 2

## 1. Executive Summary

The goal of the BI solution is to provide two complementary reporting experiences:

- A strategic dashboard for the Executive Director to monitor sales performance, revenue trends, geographical performance, and customer value.
- An operational dashboard for the Operations Manager to identify delivery bottlenecks, delayed orders, and logistics issues requiring action.

The solution uses a PostgreSQL data warehouse with Stage, Core, and Mart layers. Power BI connects to the reporting-ready Mart Star Schema.

## 2. Target Audience & User Personas

**Executive Director (Strategic)**

The Executive Director monitors overall sales performance, revenue trends, geographical results, product categories, and customer segmentation. The dashboard supports identifying high-performing markets, underperforming regions, and valuable customer segments.

**Operations Manager (Tactical)**

The Operations Manager focuses on delivery performance, delayed orders, delivery durations, and geographical bottlenecks. The dashboard helps identify operational issues requiring investigation or corrective action.

## 3. Key Metrics & KPIs

**Reporting Scope and Order Status Rules**

The Mart layer preserves all order statuses for data completeness and auditability. Core sales, customer, and delivery KPI measures explicitly exclude canceled orders through DAX filter logic rather than relying on a report-level filter.

Metrics respond to the applicable date, geographical, product-category, and customer-segment filters. RFM eligibility is defined separately from the general reporting population.

**Sales and Customer Metrics**

- **Total Revenue:** Sum of product prices and freight charges (`total_value`) for non-canceled orders.
- **Total Orders:** Distinct count of non-canceled orders (`order_id`).
- **Average Order Value (AOV):** Total Revenue / Total Orders.
- **Total Customers:** Distinct count of purchasing customers (`customer_unique_id`) represented by non-canceled transactions within the active filter context.
- **Orders per Customer:** Total Orders / Total Customers.
- **Revenue per Customer:** Total Revenue / Total Customers.
- **Average Freight:** Average total freight charge per distinct non-canceled order, rather than the average per order-item row.
- **MoM Growth %:** Month-over-month performance change based on the selected metric, using completed reporting months.
- **Regional Revenue Share %:** Revenue in the selected geographical context divided by the relevant total within the user's authorized RLS scope.

**Delivery Metrics**

- **Total Delayed Orders:** Distinct count of non-canceled orders with `delayed_flag = 1`.
- **Avg Delivery Days:** Average duration between purchase and actual customer delivery, calculated once per order.
- **Avg Delay Days:** Average number of days beyond the estimated delivery date, calculated for delayed orders.
- **SLA Eligible Orders:** Distinct count of non-canceled orders with an available delivery-timeliness classification (`delayed_flag` is not blank).
- **Delayed Order Rate %:** Total Delayed Orders / SLA Eligible Orders.
- **% On-Time Delivery:** Number of on-time orders / SLA Eligible Orders.

**SLA Eligibility and Missing Delivery Dates**

SLA Eligible Orders include distinct non-canceled orders for which both the actual customer delivery date and the estimated delivery date are available.

Orders without either date are excluded from on-time and delayed-delivery rate calculations because their delivery performance cannot be reliably classified.

SQL validation identified 96,478 delivered orders, of which 96,470 were SLA-eligible. The remaining 8 orders have a `delivered` status but lack an actual delivery date. All delivered orders have an estimated delivery date.

Across the full current warehouse history, 1,735 non-canceled orders lack an actual delivery date, including these 8 delivered-status records.

The exclusion prevents incomplete delivery information from artificially improving or worsening the reported SLA rates.

**Delivery SLA Definition**

An order is:

- **On-Time** when the actual customer delivery date is on or before the estimated delivery date.
- **Delayed** when the actual customer delivery date is after the estimated delivery date.
- **SLA-eligible** when both actual and estimated delivery dates are available.

Delivery duration, delay duration, and delayed-order flags are calculated in PostgreSQL.

The dashboard uses the following illustrative On-Time Delivery KPI thresholds:

- Green: 95% or higher.
- Orange: 90% to below 95%.
- Red: below 90%.

These thresholds are project-specific visualization conventions, not an officially published Olist SLA.

**RFM Segmentation**

RFM is calculated in Python at the `customer_unique_id` level using eligible purchase records and loaded into PostgreSQL.

- **Recency:** Days since the customer's latest eligible purchase.
- **Frequency:** Number of distinct eligible orders.
- **Monetary:** Total product value purchased, excluding freight.

Eligible purchase statuses are `approved`, `invoiced`, `processing`, `shipped`, and `delivered`.

The five customer segments are Champion, At risk, Newbies, Regulars, and Sleeping.

The complete RFM segmentation contains **94,983 customers**. Another 437 customers do not meet the eligibility criteria and retain a NULL segment rather than being assigned an artificial sixth segment.

The SQL Mart provides `current_rfm_segment` for Power BI reporting. This represents the customer's current RFM classification, not their historical segment at each transaction date.

**RFM Segment Share Metrics**

- **Customer Share %:** Distinct purchasing customers in a given RFM segment divided by distinct purchasing customers across all non-null RFM segments within the applicable reporting context.
- **Revenue Share %:** Revenue generated by a given RFM segment divided by total revenue generated across all non-null RFM segments within the applicable reporting context.

Both metrics preserve applicable date, geography, and product-category filters while removing the RFM segment filter from their denominators.

Revenue Share uses the Stage 3 reporting definition (`price + freight_value`), whereas RFM Monetary uses product prices only. This distinction is intentional.

## 4. Functional Requirements

The Power BI solution provides four main dashboard pages: three strategic views for the Executive Director and one operational view for the Operations Manager.

The report supports interactive business analysis through reusable KPIs, dynamic metric selection, synchronized filters, geographical drill-down, contextual tooltips, and order-level drill-through functionality.

### Strategic Dashboard — Executive Director

**1. KPI Overview**

The KPI Overview page provides an executive summary of sales performance through:

- Four headline KPI cards: **Total Revenue**, **Total Orders**, **Average Order Value (AOV)**, and **Month-over-Month Change %**.
- A Monthly Trend chart displaying sales performance over time.
- A metric selector allowing users to analyze Revenue, Orders, or AOV.
- A dynamic Top 10 Customer States table ranked according to the selected metric, with data bars and drill-through functionality.
- Interactive filtering by purchase date, customer state, city, product category, and RFM segment.

The Top 10 table displays the ten highest-ranking states without an Others category. Its ranking dynamically responds to the selected business metric and relevant report filters.

The Monthly Trend uses readable month-year labels and excludes incomplete or unrepresentative edge periods at the visual level without changing the overall KPI totals.

**Month-over-Month Calculation**

The MoM KPI compares the latest completed reporting month with the preceding month.

For the reference reporting period ending on September 3, 2018, the displayed **−4.12%** compares **August 2018 against July 2018**.

September 2018 is excluded from the comparison because the available source data end on September 3, 2018.

A dynamic comparison-period label identifies the months used in the calculation. The displayed months update when the reporting date context changes.

This prevents misleading comparisons between complete and incomplete reporting months.

**2. Geographic Sales Analysis**

The Geographic Sales Analysis page provides geographical and product-category insights through:

- Four KPI cards: **Total Revenue**, **Total Orders**, **Average Freight**, and **% On-Time Delivery**.
- An interactive geographical map using customer latitude and longitude from PostgreSQL.
- A location performance bar chart supporting hierarchical drill-down from Country to Region, State, and City.
- Dynamic geographical comparisons using Revenue, Orders, and AOV.
- A Top 5 Product Categories by Revenue visualization.
- A contextual map tooltip presenting geographical, sales, and delivery information.

The geographical analysis supports identifying high-performing markets, comparing regional results, and investigating geographical delivery performance.

**Geographical Map and Tooltip**

The map displays customer locations using geographical coordinates stored in the PostgreSQL Mart customer dimension.

When a user selects a state or hovers over its geographical points, the contextual tooltip provides additional information about the selected geographical area.

The tooltip includes:

- **Selected State** — the state associated with the current geographical context.
- **Top City by Orders** — the leading city in the selected state, including its number of orders.
- **Total Revenue** — revenue generated in the selected geographical context.
- **Orders in Selection** — distinct orders associated with the selection.
- **Total Customers** — distinct purchasing customers.
- **AOV** — average revenue per order.
- **Delayed Order Rate %** — percentage of SLA-eligible orders delivered late.

For example, when the state **CE (Ceará)** is selected, the tooltip identifies **Fortaleza** as the top city, with **636 orders** in the verified reporting context.

The Top City by Orders measure adds business value to the map by identifying the most important city within a selected state rather than displaying geographical points alone.

The geographical map is supplemented by a location performance chart, allowing users to compare geographical results through quantitative measures and hierarchical drill-down.

**3. Customer Segmentation RFM**

The Customer Segmentation RFM dashboard supports customer behavior analysis, segment performance evaluation, and the identification of valuable customer groups.

The page provides:

- Four KPI cards: **Total Customers**, **AOV**, **Orders per Customer**, and **Revenue per Customer**.
- A **Customers by RFM Segment** horizontal bar chart displaying the customer distribution across five RFM segments.
- A **Customer Share % vs Revenue Share %** clustered column chart comparing each segment's contribution to the customer population and total revenue.
- An **RFM Customer Behavior** table containing Avg Recency (Days), Orders per Customer, AOV, Avg RFM Monetary, and Total Revenue.
- Interactive filtering by purchase date, customer state, city, product category, and RFM segment.

The dashboard uses the SQL-derived `current_rfm_segment` attribute, which represents the customer's current RFM classification rather than their historical classification at the time of each transaction.

**Customer Population and Filtering**

The RFM page focuses on customers with an assigned RFM segment.

The complete RFM segmentation contains **94,983 customers**, while the broader non-canceled customer population contains **94,989 customers** across the full warehouse history.

The six-customer difference represents customers without an eligible RFM segment in the broader non-canceled population.

The page-level filter excluding blank `current_rfm_segment` values is intentionally retained for segment-focused analysis.

The page also responds to the selected purchase-date range and other applicable report filters.

Consequently, a shorter reporting period can produce a lower Total Customers value even when the underlying RFM segmentation remains unchanged.

For example, the previously observed difference between approximately **93.98K** customers on the RFM page and **94,989** customers in the broader reporting population was primarily caused by comparing different date ranges.

After extending the RFM reporting period to include earlier purchases, the displayed customer count increased to approximately **94.98K**.

Customer-count comparisons between dashboard pages must therefore use consistent date, geographical, and segment-filtering contexts.

**Customer Share % vs Revenue Share %**

The clustered column chart compares two complementary metrics:

- **Customer Share %:** The proportion of distinct purchasing customers belonging to each RFM segment within the segmented reporting population.
- **Revenue Share %:** The proportion of total revenue generated by each RFM segment within the applicable reporting context.

These measures allow users to assess whether a segment contributes proportionally more or less revenue than its share of customers.

The comparison preserves relevant business filters while removing the segment filter from the denominator.

Revenue Share uses the Stage 3 reporting definition of revenue, including product price and freight.

This differs from RFM Monetary, which includes product prices only.

**Customers by RFM Segment**

The horizontal bar chart displays the distribution of customers across five segments:

- Regulars
- Newbies
- Sleeping
- At risk
- Champion

The chart uses consistent segment colors and data labels to support comparisons between customer groups with substantially different populations.

**Champion Segment Interpretation**

The Champion segment contains **143 customers** in the complete RFM population, representing approximately **0.15%** of segmented customers.

Due to its small population relative to Regulars, Newbies, and Sleeping customers, the Champion bar may appear almost invisible.

This reflects the actual customer distribution rather than missing data or a visualization error.

The Customer Share % vs Revenue Share % chart provides additional insight by showing that Champions contribute a higher proportion of revenue than their customer population share.

**RFM Customer Behavior Table**

The RFM Customer Behavior table provides a detailed comparison of the five customer segments.

It includes:

- **Avg Recency (Days):** Average recency of customers within the segment.
- **Orders per Customer:** Average number of orders per purchasing customer.
- **AOV:** Average revenue generated per order.
- **Avg RFM Monetary:** Average customer monetary value calculated from product prices, excluding freight.
- **Total Revenue:** Total revenue generated by the segment, including product prices and freight charges.

The table contains an **Overall** summary row.

The Overall row displays aggregated results for the selected reporting population, using the appropriate calculation for each metric.

Average-based indicators retain their respective aggregation logic, while Total Revenue displays the overall revenue sum.

Adding Total Revenue makes each segment's absolute financial contribution directly visible, complementing the percentage-based Revenue Share chart.

**Business Value**

The RFM dashboard helps stakeholders:

- Identify customer groups contributing disproportionately to revenue.
- Compare customer concentration with financial contribution.
- Evaluate purchasing behavior across segments.
- Identify valuable customers who may benefit from retention initiatives.
- Prioritize customer engagement strategies based on both segment size and financial performance.

The dashboard supports customer-focused decision-making while maintaining consistent metric definitions and transparent reporting filters.

### Operational Dashboard — Operations Manager

**4. Delivery Performance**

The Delivery Performance dashboard supports operational monitoring, delivery performance evaluation, and the identification of geographical logistics risks.

The page provides:

- Four headline KPI cards: **Total Delayed Orders**, **% On-Time Delivery**, **Avg Delay Days**, and **Avg Delivery Days**.
- A **Delivery Risk & Impact by State** table showing Delayed Orders, Delay Rate %, and SLA Eligible Orders.
- A **Delivery Time Distribution** chart displaying the distribution of actual delivery durations.
- An **Order Volume vs Delay Rate** combination chart comparing monthly order volumes and delivery delay rates.
- An **Order Status** slicer for operational filtering.

**Delivery Risk & Impact by State**

The table identifies the ten states with the highest delayed-order rates among states with at least **100 SLA-eligible orders**.

The remaining states are grouped into an **Others** category.

The Others delay rate is calculated by dividing the combined number of delayed orders by the combined number of SLA-eligible orders across the remaining states.

It is not calculated as a simple arithmetic average of state-level delay percentages.

Conditional formatting compares each state's delayed-order rate against the overall rate:

- **Red:** Above the overall delayed-order rate.
- **Navy:** Equal to or below the overall delayed-order rate.

The table allows users to evaluate both relative delivery risk and the absolute number of affected orders.

The dynamic ranking and aggregated Others category respond to applicable business filters.

**Delivery Time Distribution**

The chart groups orders according to their actual delivery duration.

It helps identify typical fulfillment times and the proportion of orders experiencing longer delivery periods.

**Order Volume vs Delay Rate**

The combination chart compares monthly order volumes with delayed-order rates.

Monthly delay-rate values based on fewer than **100 SLA-eligible orders** are suppressed to avoid misleading conclusions from very small samples.

Order volumes remain visible even when the corresponding delayed-order rate is suppressed.

The incomplete September 2018 reporting month is excluded from the monthly trend at the visual level without changing the underlying KPI calculations.

**SLA Eligibility**

Delivery performance metrics include only non-canceled orders for which both the actual customer delivery date and the estimated delivery date are available.

Orders with missing required dates are excluded from SLA-based percentage calculations because their delivery timeliness cannot be reliably classified.

SQL validation identified **96,478 delivered orders** in the current full warehouse history, of which **96,470 were SLA-eligible**.

The remaining **8 delivered-status orders** lack an actual delivery date.

The detailed eligibility definition and validation are documented in Section 3 of the BRD.

**Operational Drill-through**

The Delivery Risk & Impact table supports drill-through to the `Order_Details Operations` page.

Users can investigate individual states and the aggregated Others category to identify the orders contributing to delivery performance results.

The operational drill-through page also supports direct opening without requiring a state to be selected beforehand.

### Navigation, Filtering, and Drill-through

The report provides:

- A consistent left-side navigation panel across the four main dashboard pages.
- Shared Customer State, City, Product Category, and RFM Segment slicers.
- An Order Date range slicer for time-based analysis.
- An additional Order Status slicer on the Delivery Performance page.
- Synchronized slicer selections across the main dashboard pages.
- Reset Filters functionality using page-specific bookmarks.
- Dynamic visual titles and contextual explanatory tooltips.
- A Help interface explaining report features and business metric definitions.
- An `Order_Details` drill-through page for exploring underlying orders from the KPI state ranking.
- An `Order_Details Operations` drill-through page for investigating states and the Others category from the Delivery Risk & Impact analysis.
- Supporting report tooltip pages for selected KPIs and geographical information.

**Order-Level Drill-through**

The `Order_Details` page allows users to investigate individual orders associated with a selected geographical state.

The order-level delivery difference is labeled **Days vs Estimate**.

- **Positive values:** Delivery after the estimated date.
- **Negative values:** Delivery before the estimated date.
- **Zero:** Delivery on the estimated date.

This label reflects both early and delayed deliveries, avoiding the misleading implication that every value represents a delay.

The `Order_Details Operations` page supports drill-through from individual states and the Others group in the Delivery Risk & Impact table.

It also supports direct page opening without requiring a preselected state.

**Help and Metric Definitions**

Contextual Help and report tooltips support interpretation of business metrics, including delivery SLA eligibility and month-over-month comparisons.

The Operations — Methodology Help content explains:

- Delayed Orders and On-Time Delivery.
- Delayed Order Rate and SLA Eligible Orders.
- Average Delay Days.
- Top 10 States and Others.
- Conditional formatting and delivery risk interpretation.
- Order Volume vs Delay Rate.
- Operational drill-through.
- Report filters and state-based access permissions.

SLA eligibility requires both an actual customer delivery date and an estimated delivery date.

Orders without the required dates cannot be classified as on-time or delayed and are excluded from SLA-based percentage calculations.

The detailed eligibility definition and SQL reconciliation are documented in Section 3.

The four main dashboard pages follow a consistent visual design, with four KPI cards per page and supporting analytical visuals tailored to the relevant business audience.

## 5. Non-Functional & Technical Requirements

### Security — Dynamic Row-Level Security

Dynamic Row-Level Security (RLS) restricts access to customer and sales data based on the authenticated user's assigned role and permitted state.

The security mapping is maintained in the PostgreSQL table `mart.dim_security`, which contains user identities, access roles, and authorized states.

The Power BI model implements the `Dynamic_Security` role using `USERPRINCIPALNAME()`.

Access rules are defined as follows:

- **Director:** Users assigned the Director role with `ALL` state access can view data across all states.
- **Manager:** Users can access their assigned state or all states when explicitly assigned `ALL`.
- **Unauthorized users:** Users without a matching security mapping cannot access customer and sales data protected by RLS.

The security table is intentionally disconnected from the Star Schema. The security rule is evaluated on `mart.dim_customer`, and the customer-to-fact relationship propagates the restrictions to `mart.fact_sales`.

RLS configuration has been implemented in Power BI Desktop. Final validation of user identities and role assignments in Power BI Service remains required before deployment.

### Data Model — Star Schema

Power BI uses the PostgreSQL Mart layer as its reporting semantic model.

The model contains:

- `mart.fact_sales` — order-item-level sales transactions, financial amounts, order statuses, delivery metrics, and date keys.
- `mart.dim_customer` — customer attributes, geographical information, SCD Type 2 history, and RFM segmentation.
- `mart.dim_product` — product attributes and categories.
- `mart.dim_seller` — seller identifiers.
- `mart.dim_date` — reporting calendar.
- `mart.dim_security` — independent user-to-state authorization mapping.

The fact table uses a grain of one row per `order_id` and `order_item_id`.

Financial reporting uses `total_value`, defined as product price plus freight value.

The model preserves historical customer versions through surrogate keys and temporal customer matching performed in PostgreSQL.

### Advanced DAX and Reporting Logic

The Power BI semantic model uses reusable DAX measures for reporting calculations, dynamic comparisons, and contextual filtering.

Key techniques include:

- **`CALCULATE` and `KEEPFILTERS`:** Apply business rules, including the explicit exclusion of canceled orders from core reporting measures.
- **`SUMX`:** Calculate Total Revenue from transaction-level values.
- **`DISTINCTCOUNT`:** Count unique orders without duplicating multi-item transactions.
- **`DIVIDE`:** Calculate ratios and percentages safely.
- **`REMOVEFILTERS`:** Establish appropriate denominator contexts for KPIs and segment-share calculations.
- **`ALLSELECTED`:** Support dynamic rankings while respecting the user's relevant selections.
- **`TREATAS`:** Apply virtual filter context to supporting analytical tables, including state-based comparisons.
- **`USERPRINCIPALNAME`:** Identify the current user for dynamic RLS.

Dynamic measures support switching between Revenue, Orders, and AOV, as well as generating corresponding visual titles.

The customer-share and revenue-share measures compare the five RFM segments while preserving applicable business filters.

Delivery measures distinguish delayed, on-time, and SLA-eligible orders. On-time classification explicitly checks `delayed_flag == 0` to avoid interpreting blank values as on-time deliveries.

SQL is responsible for calculating delivery durations, delays, and current customer RFM segmentation. DAX handles presentation-level aggregations and business analysis.

### Date Management

The reporting calendar is provided by `mart.dim_date`, with Power BI Auto date/time disabled.

The active relationship connects `mart.fact_sales[order_date_id]` with `mart.dim_date[date_id]`, allowing the date slicer to filter orders by purchase date.

Four additional relationships exist for:

- Order approval date.
- Carrier handoff date.
- Customer delivery date.
- Estimated delivery date.

These relationships remain inactive and are not currently used in dedicated `USERELATIONSHIP()` measures.

They are retained for potential future order-milestone analysis and remain an open model-cleanup decision.

### Connectivity and Refresh

Power BI imports reporting data from PostgreSQL using configurable Power Query parameters:

- `pServer` — PostgreSQL server address.
- `pDatabase` — database name.

This avoids embedding a fixed local server configuration in individual reporting queries.

The target reporting refresh frequency is daily. During local development, warehouse loading and Power BI refreshes are performed manually.

### Performance and Usability

The report is designed to support responsive, understandable analysis through:

- A reporting-oriented Star Schema.
- Reusable DAX measures.
- Clear and consistent KPI definitions.
- Four main dashboard pages with consistent visual design.
- Synchronized filters and page-specific Reset functionality.
- Dynamic visual titles and contextual tooltips.
- Drill-through pages for detailed order investigation.
- Minimum sample-size rules for selected delivery-rate comparisons.

The report suppresses monthly delay-rate values when fewer than 100 SLA-eligible orders are available.

Formal Power BI performance benchmarking and production refresh testing have not yet been completed.

## 6. Known Limitations and Pending Improvements

- **Geolocation Data Quality:** Four current customer records contain coordinates outside the expected Brazilian boundaries. These source-data outliers require correction in the upstream ETL process and subsequent warehouse refresh.

- **Power BI Service Validation:** The geographical map and Dynamic Row-Level Security have been configured in Power BI Desktop. Their behavior, including security role membership and user identity mapping, requires final validation in Power BI Service.

- **Inactive Date Relationships:** Four additional date relationships for order approval, carrier handoff, customer delivery, and estimated delivery remain inactive. They may be used in future milestone-date analysis or removed if not required by the reporting scope.

- **Geographic Cross-Page Navigation:** The geographical drill-down and synchronized slicers are functional. Direct transfer of a selected map location to Delivery Performance remains a potential enhancement.

STEP 3 - Wireframing & Mockups

Low-fidelity wireframes were initially prepared for four business dashboard pages:

- KPI Overview
- Map / Drill-down
- RFM Segmentation
- Operations Dashboard

These initial wireframes established the proposed layout of KPI cards, navigation, filters, and supporting visualizations.

During Power BI development, the design evolved into four final report pages:

- **KPI Overview** — four KPI cards, a monthly trend chart, and a dynamic Top 10 Customer States ranking.
- **Geographic Sales Analysis** — geographical drill-down, customer-location map, product-category analysis, and contextual map tooltips.
- **Customer Segmentation RFM** — four KPI cards, customer distribution by segment, Customer Share % vs Revenue Share %, and customer behavior analysis.
- **Delivery Performance** — four operational KPI cards, Delivery Risk & Impact by State, Delivery Time Distribution, and Order Volume vs Delay Rate.

The revised design includes consistent navigation, synchronized business filters, a purchase-date slicer, page-specific Reset functionality, and supporting drill-through pages.

The initial wireframes are retained as evidence of the design process. Updated wireframes should reflect the implemented report rather than earlier visualization concepts.

The final Power BI report remains the source of truth for dashboard functionality and visual layout.


STEP 4

### Pattern 1 — Customer revenue contribution differs from customer share

The Customer Segmentation RFM dashboard shows that customer population and revenue contribution are not distributed identically across RFM segments.

In the reference reporting period:

- Regulars account for approximately 50.73% of segmented customers and 49.94% of segment revenue.
- Champions represent only 0.15% of customers but contribute approximately 0.50% of revenue.
- At-risk customers represent 1.11% of customers but contribute approximately 2.32% of revenue.

**Hypothesis:**

Smaller, higher-value customer segments can contribute disproportionately to revenue. Customer count alone is therefore insufficient to evaluate the business importance of each segment.

**Business Action:**

Prioritize retention campaigns and customer-experience initiatives for valuable customer groups, particularly At-risk and Champion customers. Use both Customer Share % and Revenue Share % when allocating marketing and retention resources.

### Pattern 2 — Delivery risk should be evaluated using both delay rate and affected order volume

The Delivery Performance dashboard shows 6,509 delayed orders among 95,453 SLA-eligible orders in the reference reporting period.

This corresponds to approximately 6.82% delayed orders and 93.18% on-time deliveries.

The Delivery Risk & Impact by State table identifies states with relatively high delay rates while also showing the number of affected orders and eligible orders.

A high delay rate does not necessarily indicate the largest operational impact. States with lower delay percentages but substantially higher order volumes may account for more delayed deliveries.

**Hypothesis:**

Delivery problems differ across geographical markets. Some states exhibit higher relative delivery risk, while others create greater absolute operational impact due to their order volume.

**Business Action:**

Use delay rate and delayed-order volume together to prioritize logistics improvements. Investigate high-risk states, monitor high-volume markets, and use the monthly Order Volume vs Delay Rate chart to identify periods requiring additional operational attention.

STEP 5

## SQL vs Power BI Validation

Three business metrics were independently validated using SQL queries executed against the PostgreSQL Mart layer:

- Total Revenue
- Total Orders
- Total Delayed Orders

The validation uses consistent order-status exclusions and reporting filter contexts. Canceled orders are excluded from the corresponding business measures.

### Test 1 — Dashboard Reporting Period

**Filter context:** February 1, 2017 – September 3, 2018.

| Metric | SQL Result |
|---|---:|
| Total Revenue | R$ 15,546,929.74 |
| Total Orders | 97,119 |
| Total Delayed Orders | 6,509 |

SQL and Power BI results matched for all three metrics.

### Test 2 — State and Date Filtering

**Filter context:** Customer State = SP, March 1–31, 2018.

| Metric | SQL Result |
|---|---:|
| Total Revenue | R$ 436,641.43 |
| Total Orders | 3,007 |
| Total Delayed Orders | 280 |

Power BI results matched the SQL order counts and delayed-order counts. Revenue matched at the displayed KPI-card precision.

These tests demonstrate that the dashboard responds correctly to the tested date and geographical filters.

### Historical Data Reconciliation Note

A separate comparison between earlier validation evidence and the current warehouse identified the following differences:

| Metric | Earlier Validation | Current SQL |
|---|---:|---:|
| Total Revenue | R$ 15,737,887.53 | R$ 15,737,667.52 |
| Total Orders | 98,208 | 98,205 |
| Total Delayed Orders | 6,534 | 6,534 |

The difference of three orders and R$ 220.01 remains under investigation.

Additional SQL checks confirmed that joining the current fact table to the relevant dimensions does not explain this discrepancy. The historical comparison is therefore documented separately from the successfully validated dashboard reporting period.

### Validation Evidence

Supporting evidence consists of SQL query results from DBeaver and corresponding Power BI dashboard screenshots.

The validation distinguishes exact numerical agreement from agreement at the rounded display precision of Power BI KPI cards.